In [ ]:
import requests
import time
import base64
import json
from PIL import Image
from io import BytesIO
from datetime import datetime
from colorama import Fore, Style, init
import os
import sys

# === INIT ===
init(autoreset=True)

# Set terminal title (works on Windows)
os.system("title DiffusionCraft Worker")

# === CONFIGURATION ===
GATEWAY_BASE_URL = "https://vwx6lrkyh4.execute-api.us-east-1.amazonaws.com/prod"
READ_PROMPT_URL = f"{GATEWAY_BASE_URL}/readPromptFromQueue"  
UPLOAD_IMAGE_URL = f"{GATEWAY_BASE_URL}/uploadImageData"


# === LOAD YOUR DIFFUSION PIPELINE HERE ===
def generate_fake_image(prompt):
    img = Image.new("RGB", (512, 512), (42, 42, 42))
    buffer = BytesIO()
    img.save(buffer, format="PNG")
    return buffer.getvalue()

# === MAIN WORKER LOOP ===
def worker_loop():
    print(Fore.CYAN + Style.BRIGHT + f"[{datetime.now()}] 🚀 DiffusionCraft worker is running...")

    while True:
        try:
            # Step 1: Pull message from queue
            response = requests.get(READ_PROMPT_URL)
            if response.status_code != 200:
                print(Fore.RED + f"❌ Failed to fetch prompt: {response.text}")
                time.sleep(CHECK_INTERVAL)
                continue

            data = response.json()
            if "prompt" not in data:
                print(Fore.YELLOW + f"[{datetime.now()}] ⏳ No prompts in queue.")
                time.sleep(CHECK_INTERVAL)
                continue

            prompt = data["prompt"]
            user_sub = data["userSub"]
            image_id = data["imageId"]

            print(Fore.BLUE + f"[{datetime.now()}] 🎨 Generating image for: '{prompt}'")

            # Step 2: Generate image
            image_bytes = generate_fake_image(prompt)
            image_base64 = base64.b64encode(image_bytes).decode("utf-8")

            # Step 3: Upload image
            upload_payload = {
                "userSub": user_sub,
                "imageId": image_id,
                "imageBase64": image_base64
            }

            upload_resp = requests.post(UPLOAD_IMAGE_URL, json=upload_payload)
            if upload_resp.status_code == 200:
                print(Fore.GREEN + f"[{datetime.now()}] ✅ Image for '{prompt}' uploaded successfully.")
            else:
                print(Fore.RED + f"[{datetime.now()}] ❌ Upload failed: {upload_resp.text}")

        except Exception as e:
            print(Fore.RED + f"🔥 Unexpected error: {str(e)}")

        time.sleep(CHECK_INTERVAL)

# === RUN ===
if __name__ == "__main__":
    try:
        worker_loop()
    except KeyboardInterrupt:
        print(Fore.MAGENTA + "\n👋 Worker stopped by user.")
        sys.exit(0)


In [7]:
import torch
print("CUDA Available:", torch.cuda.is_available())
print("Device Name:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "None")
print("CUDA Version:", torch.version.cuda)


CUDA Available: False
Device Name: None
CUDA Version: None
